# Quantum Reservoir Prep

Builds multi-circuit quantum reservoir tracks from the Track B predictors, in the spirit of the Chen et al. quantum recurrent networks (QLSTM, [arXiv:2009.01783](https://arxiv.org/abs/2009.01783); QRNN reservoir computing, [arXiv:2206.11283](https://arxiv.org/abs/2206.11283)). Each combination of encoding strategy × result getter × target is saved as a dataframe track, `data/processed/track_<TRACK>_<target>.parquet`, ready for `TRACK_CONFIG` in `02_evaluation_loop`.

How a window becomes a feature row:

- At month t of a trailing `WINDOW`-month window, the circuit encodes the visible vector x_t (that month's Track B predictors) together with the hidden vector h_{t-1}, as Chen's QRNN encodes v_t = [x_t, h_{t-1}].
- A fixed, untrained random `efficient_su2` ansatz acts on all qubits, and the result getter's output becomes h_t. The hidden state starts at zero.
- The hidden state is passed between circuits classically, so each month is one circuit run (`multi_circuit_quantum_reservoir`). The final h of the window is that month's feature row.
- By default there are as many hidden features as visible ones (`NUM_HIDDEN = None`), so the track has the same width as Track B.

As with the QRC track from `scripts/build_quantum_tracks.py`, a row depends only on its own trailing window and the reservoir is never trained. Rows don't change between folds, so the tracks can be precomputed once. The first `WINDOW - 1` months have no full window and are left out.

This notebook only holds configuration. The logic lives in `src/features/quantum/`:

- `encoding_strategy/`: the `su`, `heisenberg` and `zz` encodings used by `quantum_data_prep`
- `result_getters/`: the readout that becomes the hidden state
- `multi_circuit_quantum_reservoir.py`: the recurrence

## 1. Environment setup

In [ ]:
import os
import sys
from pathlib import Path

COLAB = False  # set to True when running in Google Colab

if COLAB:
    REPO_URL  = "https://github.com/manuelarceaguirre/capstone-quantum.git"
    BRANCH    = "clean"
    REPO_ROOT = Path("/content/capstone-quantum")

    %cd /content
    !rm -rf capstone-quantum
    !git clone -b {BRANCH} {REPO_URL}
    %cd /content/capstone-quantum
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime
    sys.path.insert(0, str(REPO_ROOT))
else:
    # local: walk up from the working directory to the repo root;
    # src/ is importable via `pip install -e .` (see environment.yml)
    REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

print("Repo root:", REPO_ROOT)

In [ ]:
import json
import time
import warnings

import numpy as np
import pandas as pd
from qiskit.circuit.library import efficient_su2
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import EstimatorV2, SamplerV2

from src.features.quantum import multi_circuit_quantum_reservoir
from src.features.quantum.encoding_strategy import heisenberg_encoding_strategy, su_encoding_strategy
from src.features.quantum.result_getters import (
    pce_result_getter,
    single_bit_sampler_result_getter,
    single_qubit_pauli_result_getter,
)

warnings.filterwarnings("ignore")

## 2. Configuration

A note on scrambling: the reservoir feeds its own output back `WINDOW` times, so an ansatz that scrambles the state too well leaves features that barely vary between windows. In early tests a heavier entangling block cut the across-window spread of the features several-fold, close to the shot noise (about 0.008 at 4096 shots). Hence `ANSATZ_REPS = 1`.

Width: each circuit encodes visible + hidden values (20 or 24 here). The `su` encoding packs them onto 10 or 12 qubits, about 3 min per target with the `z` getter. The `heisenberg` and `zz` encodings use one qubit per value plus one (21 to 25 qubits). That costs about 0.5 s per circuit on Aer, or more than 1.5 h per target.

In [ ]:
# --- Paths ---
DATA_DIR = REPO_ROOT / "data" / "processed"
INPUT    = "track_b_{target}.parquet"   # Track B predictor set per target

# --- What to run ---
TARGET_LIST      = ["INDPRO", "PAYEMS", "CPIAUCSL", "S&P 500"]
ACTIVE_ENCODINGS = ["su"]   # any keys of ENCODING_REGISTRY
ACTIVE_GETTERS   = ["z"]    # any keys of GETTER_REGISTRY; "z" is the Chen QRNN readout
MAX_WINDOWS      = None   # set to a small int for a quick smoke test (output is then NOT saved)
RESUME           = False  # True: skip combinations whose track file already exists

# --- Reservoir ---
WINDOW     = 12     # months per window = circuits run in sequence per feature row
NUM_HIDDEN = None   # hidden features; None = as many as visible features

# --- Aer simulator ---
SEED                = 42
ESTIMATOR_PRECISION = 0.01   # target standard error of each expectation value (estimator getters)
SHOTS               = 4096   # shots per circuit evaluation (sampler getters)
NOISE_BACKEND       = None   # e.g. qiskit_ibm_runtime.fake_provider.FakeTorino() to add its noise model

# --- Encoding / ansatz / getter settings ---
NUM_TROTTER_STEPS = 6    # Heisenberg and ZZ encodings
ANSATZ_REPS       = 1    # layers of the random efficient_su2 ansatz; more layers scramble the state (see note above)
PCE_OPERATORS     = 2    # Paulis per correlator (pce getter)
VALIDATE_WINDOWS  = 10   # windows re-run exactly (no shot noise) to check the Aer results

# --- Encoding registry: name -> factory(n_values) ---
# n_values = visible + hidden features. Inputs are min-max scaled causally below and the
# reservoir rescales the hidden state to [0, 1], so every encoding uses scale=False.
ENCODING_REGISTRY = {
    "su":        lambda n: su_encoding_strategy(n_qubits=int(np.ceil(n / 2)), reps=1, su2_gates=["ry", "rx"],
                                                 entanglement="full", scale=False),
    "heisenberg": lambda n: heisenberg_encoding_strategy(n_features=n, num_trotter_steps=NUM_TROTTER_STEPS,
                                                         seed=SEED, scale=False),
    "zz":         lambda n: heisenberg_encoding_strategy(n_features=n, num_trotter_steps=NUM_TROTTER_STEPS,
                                                         paulis=("ZZ",), seed=SEED, scale=False),
}


def random_su2(n_qubits):
    """Reservoir ansatz: efficient_su2 with fixed random angles."""
    ansatz = efficient_su2(n_qubits, reps=ANSATZ_REPS)
    rng = np.random.default_rng(SEED)
    return ansatz.assign_parameters(2 * np.pi * rng.random(ansatz.num_parameters))


# --- Result getter registry: name -> (primitive kind, hidden bounds, factory(n_qubits, n_hidden, primitive)) ---
# The getter's output is the hidden state, so it must return exactly n_hidden columns; a factory
# returns None when it can't, and that combination is skipped. Hidden bounds are the range of
# the getter's outputs, mapped to [0, 1] before they are encoded. primitive=None gives exact simulation.
GETTER_REGISTRY = {
    # P(1) on each of the first n_hidden qubits, i.e. the Chen QRNN single-qubit <Z> readout up to (1 - <Z>) / 2
    "z":     ("sampler", (0.0, 1.0),
              lambda nq, nh, p: single_bit_sampler_result_getter(nh, sampler=p, shots=SHOTS) if nh <= nq else None),
    "pce":   ("estimator", (-1.0, 1.0),
              lambda nq, nh, p: pce_result_getter(nq, nh, PCE_OPERATORS, estimator=p)),
    # <X>, <Y>, <Z> on the first n_hidden / 3 qubits
    "pauli": ("estimator", (-1.0, 1.0),
              lambda nq, nh, p: single_qubit_pauli_result_getter(nh // 3, estimator=p)
                                if nh % 3 == 0 and nh // 3 <= nq else None),
}


def track_name(encoding, getter):
    """Track key used in TRACK_CONFIG and in the file name."""
    return f"MCR_{encoding}_{getter}"


print(f"Targets:   {TARGET_LIST}")
print(f"Encodings: {ACTIVE_ENCODINGS}")
print(f"Getters:   {ACTIVE_GETTERS}")
print(f"Reservoir: {WINDOW}-month windows, hidden = {'visible' if NUM_HIDDEN is None else NUM_HIDDEN}")
print(f"Aer:       precision {ESTIMATOR_PRECISION}, {SHOTS} shots, noise {'off' if NOISE_BACKEND is None else NOISE_BACKEND.name}")

## 3. Load and scale inputs

Track B inputs are min-max scaled with expanding (causal) statistics, the same way `scripts/build_quantum_tracks.py` and `quantum_data_prep` scale them: each month uses only the minimum and maximum seen up to that month. Every month with a full trailing window gets one feature row.

In [ ]:
def causal_minmax(df: pd.DataFrame) -> pd.DataFrame:
    """Scale each column to [0, 1] using min/max over rows up to and including t."""
    lo, hi = df.expanding().min(), df.expanding().max()
    return ((df - lo) / (hi - lo).replace(0, np.nan)).fillna(0.5).clip(0, 1)


def trailing_windows(n_rows: int, window: int) -> list[list[int]]:
    """Row-index sequences [t-window+1, ..., t] for every t with a full window."""
    return [list(range(t - window + 1, t + 1)) for t in range(window - 1, n_rows)]


inputs, windows = {}, {}
for target in TARGET_LIST:
    raw = pd.read_parquet(DATA_DIR / INPUT.format(target=target)).sort_index()
    inputs[target] = causal_minmax(raw)
    windows[target] = trailing_windows(len(raw), WINDOW)[:MAX_WINDOWS]
    print(f"  {target:<10} {raw.shape[1]:>2} features  {len(windows[target])} windows  "
          f"{raw.index[WINDOW - 1]:%Y-%m} -> {raw.index[WINDOW - 2 + len(windows[target])]:%Y-%m}")

## 4. Aer primitives

Both primitives run locally on the Aer simulator through `qiskit_ibm_runtime`'s local mode, the same interface as IBM hardware. Step t of every window shares one circuit, so each step is one `run()` call and a whole track is `WINDOW` calls.

In [ ]:
simulator = (AerSimulator(seed_simulator=SEED) if NOISE_BACKEND is None
             else AerSimulator.from_backend(NOISE_BACKEND, seed_simulator=SEED))
PRIMITIVES = {
    "estimator": EstimatorV2(mode=simulator, options={"default_precision": ESTIMATOR_PRECISION}),
    "sampler":   SamplerV2(mode=simulator, options={"default_shots": SHOTS}),
}
print(simulator)

## 5. Build the tracks

Loops encodings × getters × targets, saving each track file as soon as it is built, so an interrupted run keeps what it finished (rerun with `RESUME = True`).

In [ ]:
def reservoir_parts(enc_name, getter_name, n_visible, primitive):
    """Encoding, ansatz, getter and hidden bounds of one reservoir (getter is None if it can't fit)."""
    n_hidden = n_visible if NUM_HIDDEN is None else NUM_HIDDEN
    encoding = ENCODING_REGISTRY[enc_name](n_visible + n_hidden)
    ansatz = random_su2(encoding.n_qubits)
    _, bounds, make_getter = GETTER_REGISTRY[getter_name]
    return n_hidden, encoding, ansatz, make_getter(encoding.n_qubits, n_hidden, primitive), bounds


combos = [(e, g, t) for e in ACTIVE_ENCODINGS for g in ACTIVE_GETTERS for t in TARGET_LIST]
log, built = [], {}
t_start = time.time()

for k, (enc_name, getter_name, target) in enumerate(combos, 1):
    track = track_name(enc_name, getter_name)
    path = DATA_DIR / f"track_{track}_{target}.parquet"
    data, series = inputs[target], windows[target]
    kind = GETTER_REGISTRY[getter_name][0]
    n_hidden, encoding, ansatz, getter, bounds = reservoir_parts(
        enc_name, getter_name, data.shape[1], PRIMITIVES[kind])
    entry = {"track": track, "encoding": enc_name, "getter": getter_name, "target": target, "qubits": encoding.n_qubits, "visible": data.shape[1], "hidden": n_hidden}

    if getter is None:
        log.append({**entry, "status": f"skipped: {getter_name} can't return {n_hidden} hidden features"})
        continue
    if RESUME and MAX_WINDOWS is None and path.exists():
        log.append({**entry, "status": "exists (resume)", "features": pd.read_parquet(path).shape[1]})
        continue

    t0 = time.time()
    features = multi_circuit_quantum_reservoir(
        data=data, series_indicies=series, num_hidden_features=n_hidden,
        ansatz=ansatz, encoding_strategy=encoding, result_getter=getter, hidden_bounds=bounds,
    )
    features.index = data.index[[s[-1] for s in series]]
    features.columns = [f"{track}_{c}" for c in features.columns]
    seconds = time.time() - t0

    built[(track, target)] = features
    if MAX_WINDOWS is None:
        features.to_parquet(path)
    log.append({**entry, "status": "built" if MAX_WINDOWS is None else "built (smoke test, not saved)",
                "features": features.shape[1], "seconds": round(seconds, 1)})

    elapsed = time.time() - t_start
    print(f"  [{k}/{len(combos)}] {track:<24} {target:<10} {encoding.n_qubits:>2} qubits  "
          f"{features.shape[1]:>3} features  {seconds:6.1f}s  · {elapsed/60:.1f} min elapsed")

log = pd.DataFrame(log)
print(f"Done in {(time.time() - t_start)/60:.1f} min")
log

## 6. Validate against exact simulation

Re-runs the first `VALIDATE_WINDOWS` windows of every track exactly (no shot noise) and reports the largest difference from the Aer features. Shot noise is fed back through the hidden state, so expect somewhat more than a single circuit would give, which is a few times `ESTIMATOR_PRECISION` for the estimator getters and about 1/√`SHOTS` for the sampler getters. With a noise model the gap also includes hardware noise.

A large max with a small mean can be the reservoir itself rather than a bug. In an earlier build with the su2 ansatz, max gaps reached 0.16 to 0.65 while a single circuit's gap was normal (0.026). Two Aer runs with different seeds disagreed with each other just as much, so the reservoir amplifies shot noise through the recurrence for some features and windows.

In [ ]:
rows = []
for (track, target), features in built.items():
    entry = log.loc[(log["track"] == track) & (log["target"] == target)].iloc[0]
    data = inputs[target]
    n_hidden, encoding, ansatz, getter, bounds = reservoir_parts(
        entry["encoding"], entry["getter"], data.shape[1], None)
    exact = multi_circuit_quantum_reservoir(
        data=data, series_indicies=windows[target][:VALIDATE_WINDOWS], num_hidden_features=n_hidden,
        ansatz=ansatz, encoding_strategy=encoding, result_getter=getter, hidden_bounds=bounds,
    ).to_numpy()
    diff = np.abs(features.iloc[:VALIDATE_WINDOWS].to_numpy() - exact)
    rows.append({"track": track, "target": target, "max |aer - exact|": diff.max(), "mean |aer - exact|": diff.mean()})

validation = pd.DataFrame(rows).set_index(["track", "target"]).unstack("target")
validation.round(4)

## 7. Register the tracks

Writes `data/processed/quantum_reservoir_tracks.json` (track names, settings and run log) and prints the `TRACK_CONFIG` entries to paste into `02_evaluation_loop`. Add the names to `QUANTUM_TRACKS` in `02_evaluation_loop` and `03_plot_results` too.

In [ ]:
saved = log[log["status"].isin(["built", "exists (resume)"])]
tracks = sorted(saved["track"].unique())

if MAX_WINDOWS is None:
    manifest = {
        "built_at": pd.Timestamp.now(tz="UTC").isoformat(),
        "input": INPUT, "scaling": "causal min-max (expanding)",
        "settings": {"window": WINDOW, "num_hidden": NUM_HIDDEN, "seed": SEED,
                     "estimator_precision": ESTIMATOR_PRECISION, "shots": SHOTS,
                     "noise_backend": None if NOISE_BACKEND is None else NOISE_BACKEND.name,
                     "ansatz": "efficient_su2 (random, fixed)", "ansatz_reps": ANSATZ_REPS,
                     "num_trotter_steps": NUM_TROTTER_STEPS,
                     "pce_operators": PCE_OPERATORS},
        "tracks": tracks,
        "log": json.loads(log.to_json(orient="records")),
    }
    (DATA_DIR / "quantum_reservoir_tracks.json").write_text(json.dumps(manifest, indent=2))

print("# paste into TRACK_CONFIG in 02_evaluation_loop")
for track in tracks:
    print(f'    "{track}": {{t: {{"type": "dataframe", "path": DATA_DIR / f"track_{track}_{{t}}.parquet"}} for t in TARGET_LIST}},')
print(f"\n# QUANTUM_TRACKS += {tracks}")